#### RAG (Retrieval-Augmented Generation)
- 모델의 학습 데이터에 포함되지 않은 데이터를 사용 (환각 방지)
- **외부 데이터**를 검색(retrieval)한 후, 생성(generation) 단계에서 LLM에 전달
- 모델은 주어진 컨텍스트나 질문에 더 적합하고 풍부한 정보를 반영한 답변을 생성
- 논문: https://arxiv.org/abs/2005.11401

## 0. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add langchain langchain-openai langchain_community tiktoken faiss-cpu

### 2) OpenAI 인증키 설정
https://openai.com/

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 키 값은 출력하지 않고 설정 여부만 확인
for key_name in ("OPENAI_API_KEY", "UPSTAGE_API_KEY"):
    print(f"{key_name}: {'설정됨' if os.getenv(key_name) else '없음'}")

### 1. RAG 파이프라인 개요
* Load Data - Text Split - Indexing - Retrieval - Generation

##### Step 1: Load Data
* 1. WebBaseLoader를 사용하여 웹 페이지 데이터 가져오기

In [ ]:
import os
from langchain_community.document_loaders import WebBaseLoader

# 웹 요청을 위한 USER_AGENT 환경 변수 설정 (필요한 경우)
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

# 환경 변수 확인
print(f"현재 설정된 USER_AGENT: {os.environ.get('USER_AGENT')}")

# 웹페이지 URL 지정  https://ko.wikipedia.org/wiki/축구_경기_규칙
url = 'https://ko.wikipedia.org/wiki/%EC%B6%95%EA%B5%AC_%EA%B2%BD%EA%B8%B0_%EA%B7%9C%EC%B9%99'

# WebBaseLoader 초기화 및 데이터 로드
loader = WebBaseLoader(url)
docs = loader.load()

# 로드된 문서 확인
print(type(docs), len(docs))
print(docs)
print(type(docs[0]))  # <class 'langchain_core.documents.Document'>


In [ ]:
print(len(docs[0].page_content))
print(docs[0].page_content[5000:5500])
print(docs[0].metadata)

##### Step 2: 문서 분할(Splitting)

* WebBaseLoader를 사용하여 웹 페이지에서 가져온 데이터를 RAG 시스템에서 효율적으로 활용하기 위해 작은 청크(chunks)로 분할합니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)

# 문서 분할
splits = text_splitter.split_documents(docs)

# 분할된 문서 확인
print(type(splits), len(splits))  # 총 몇 개의 청크가 생성되었는지 확인
print(type(splits[0]))
print(splits[0].page_content[:80])  # 첫 번째 청크의 일부 출력

In [ ]:
# 열번째 청크의 내용 출력
print(splits[10].page_content[:20])
# 열번째 청크의 메타데이터 출력
print(splits[10].metadata)

##### Step 3: 벡터 DB에 저장 및 검색
* Indexing (Texts -> Embedding -> Store)

In [ ]:
from langchain_community.vectorstores import FAISS  # 벡터 저장소 라이브러리
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
print(embeddings)

# 1. FAISS 벡터 저장소 생성
# - documents: 텍스트 데이터를 벡터화 하여 저장할 문서 리스트
# - embedding: 문서를 벡터로 변환하는 Upstage Embeddings 모델 사용
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

# 2. 유사 문서 검색 (Similarity Search)
# - "경기장 표시에 대해서 설명해주세요."라는 쿼리에 대해,
# - FAISS 벡터 저장소에서 가장 유사한 문서를 검색함.
docs = vectorstore.similarity_search("경기장 표시에 대해서 설명해주세요.")

# 3. 검색된 문서의 타입과 개수 출력
print(type(docs), len(docs))
# 4. 검색된 첫 번째 문서 내용 출력
print(docs[0].page_content)

##### Step 4: RAG Pipeline을 이용한 질의응답 시스템 구축

In [ ]:
from langchain_upstage import ChatUpstage

from langchain_core.prompts import ChatPromptTemplate  # 프롬프트 템플릿
from langchain_core.runnables import RunnablePassthrough  # 입력을 그대로 전달하는 유틸리티
from langchain_core.output_parsers import StrOutputParser  # LLM 응답을 문자열로 변환하는 파서
from pprint import pprint

# 검색 개수 제한 설정
# - 벡터 저장소(vectorstore)에서 관련성이 높은 문서 최대 3개를 검색하도록 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  

def format_docs(docs):
    # 검색된 청크 전체를 출처와 함께 합침 (앞 300자만 쓰면 청크(3000자) 뒤쪽 내용이 LLM 에 전달되지 않음)
    summaries = [f"출처: {doc.metadata.get('source', '알 수 없음')}\n" + doc.page_content for doc in docs]
    return "\n\n".join(summaries)

# 프롬프트 템플릿 설정
# - 모델이 주어진 `context`(검색된 문서)만을 참고하여 질문에 답변하도록 유도하는 프롬프트 템플릿
# - {context}: 검색된 문서 요약이 삽입될 자리
# - {question}: 사용자의 질문이 삽입될 자리
template = '''당신은 제공된 컨텍스트를 기반으로 질문에 답하는 AI 어시스턴트입니다. 
반드시 컨텍스트 내 정보를 활용하여 정확하고 신뢰할 수 있는 답변을 제공하세요.

[컨텍스트]
{context}

[질문]
{question}

[답변]
'''

# - 위에서 정의한 템플릿을 사용하여 LangChain의 프롬프트 객체 생성
prompt = ChatPromptTemplate.from_template(template)  

# LLM 모델 설정
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm)


# RAG 체인 설정
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}  
    # - `retriever`를 통해 검색된 문서를 `format_docs()` 함수로 가공하여 `context`로 전달
    # - `question`은 변형 없이 그대로 전달
    | prompt  # - 위에서 정의한 `ChatPromptTemplate`을 적용
    | llm   # - Upstage solar-pro 모델을 사용해 응답 생성
    | StrOutputParser()  # - 모델의 응답을 문자열로 변환
)

# 실행 (사용자 질문을 입력으로 받아 RAG 체인 실행)
# - "경기장 표시에 대해서 설명해주세요."라는 질문을 LLM에 전달하여 답변을 생성
response = rag_chain.invoke("경기장 표시에 대해서 설명해주세요.")  

# 최종 응답 출력
print(f" 모델 응답:\n")
pprint(response)

In [ ]:
query = "선수의 장비에 대하여 설명해 주세요."

response = rag_chain.invoke(query)  

# 최종 응답 출력
print(f" 모델 응답:\n")
pprint(response)

### 개선된 Source

#### LCEL 파이프라인 연결 구조

`rag_chain`은 LCEL(LangChain Expression Language)의 파이프 연산자(`|`)로 각 단계를 연결한 체인입니다. 앞 단계의 출력이 다음 단계의 입력이 됩니다.

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | generate_chain          # generate_chain = prompt | model | StrOutputParser()
)
```

```
질문(str) ─┬─ retriever ─→ 청크 리스트 ─ format_docs ─→ context(str) ─┐
           │                                                          ├→ {"context", "question"}
           └─ RunnablePassthrough ───────────────→ question(str) ─────┘
                                                                         │
   답변(str) ◀─ StrOutputParser ◀─ model(AIMessage) ◀─ prompt(메시지) ◀──┘
```

| 단계 | 구성 요소 | 입력 → 출력 |
|---|---|---|
| 1 | `rag_chain.invoke("질문")` | 질문 문자열이 딕셔너리의 두 항목에 **같은 값으로** 전달됨 |
| 2-a | `retriever` | 질문 → 유사한 청크 `Document` 리스트 (`TOP_K`개) |
| 2-b | `format_docs` | `Document` 리스트 → `[문서 N] 출처/내용` 형태의 문자열 (`context`) |
| 2-c | `RunnablePassthrough()` | 질문을 변형 없이 그대로 전달 (`question`) |
| 3 | `prompt` | `{"context", "question"}` → 템플릿에 값을 채운 프롬프트 메시지 |
| 4 | `model` (`ChatUpstage`) | 프롬프트 → `AIMessage` (LLM 응답) |
| 5 | `StrOutputParser()` | `AIMessage` → 답변 문자열 |

**이해해 둘 점**
- **딕셔너리는 병렬 실행**: 체인 안의 `{...}`는 `RunnableParallel`로 자동 변환되어, 같은 입력(질문)으로 `context`와 `question`을 각각 계산합니다.
- **함수도 체인에 연결 가능**: `retriever | format_docs`처럼 일반 함수(`format_docs`)가 `RunnableLambda`로 자동 변환되어 파이프에 들어갑니다.
- **`generate_chain`을 따로 만든 이유**: 검색 결과(`docs`)를 이미 가지고 있을 때는 `generate_chain.invoke({"context": ..., "question": ...})`로 **검색 없이 답변 생성만** 실행할 수 있습니다. `ask_question()`이 이 방식으로 검색을 한 번만 수행합니다.
- 체인 전체를 `rag_chain.invoke(질문)` 한 줄로 실행할 수 있어 각 단계를 교체하기 쉽습니다. (예: `model`만 다른 LLM으로 변경)

In [ ]:
# 개선된 RAG 파이프라인 - 축구 규칙 질의응답 시스템
# 흐름: 웹 로드 → 분할 → 임베딩/FAISS 저장 → 검색(Retriever) → 프롬프트 → LLM 답변

import os
from dotenv import load_dotenv

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

# ── 설정 (값을 바꿀 때 이곳만 수정) ──────────────────
URL = "https://ko.wikipedia.org/wiki/%EC%B6%95%EA%B5%AC_%EA%B2%BD%EA%B8%B0_%EA%B7%9C%EC%B9%99"
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
CHUNK_SIZE = 1000          # 기존 3000 → 1000 (더 세밀한 검색)
CHUNK_OVERLAP = 200        # 청크 사이에 겹치는 글자 수 (문맥 단절 방지)
TOP_K = 6                  # 검색할 청크 수 (기존 3 → 6)
EMBEDDING_MODEL = "solar-embedding-1-large"
CHAT_MODEL = "solar-pro3"
UPSTAGE_BASE_URL = "https://api.upstage.ai/v1"

# 환경 변수 로드 및 API 키 확인 (키 값은 출력하지 않음)
load_dotenv()
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
os.environ["USER_AGENT"] = USER_AGENT  # 웹 요청 시 사용할 User-Agent

print("=== 개선된 RAG 파이프라인 시작 ===\n")

# Step 1: 웹 데이터 로드
print(" Step 1: 웹페이지 데이터 로딩...")
docs = WebBaseLoader(URL).load()   # 페이지 전체가 Document 1개로 로드됨
print(f" 로드된 문서 수: {len(docs)}")
print(f" 전체 텍스트 길이: {len(docs[0].page_content):,} 문자\n")

# Step 2: 문서 분할
print(" Step 2: 문서 분할 (개선된 설정)...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],  # 앞쪽 구분자부터 시도 (문단 → 줄 → 문장 → 단어)
)
splits = text_splitter.split_documents(docs)
print(f" 분할된 청크 수: {len(splits)}")
print(f" 첫 번째 청크 예시: {splits[0].page_content[:100]}...\n")

# Step 3: 벡터 저장소 생성 및 검색기 설정
print(" Step 3: 벡터 저장소 생성 및 검색 설정...")
embedding = UpstageEmbeddings(model=EMBEDDING_MODEL)
vectorstore = FAISS.from_documents(documents=splits, embedding=embedding)  # 청크 → 벡터 변환 후 저장 (임베딩 API 호출)
retriever = vectorstore.as_retriever(
    search_type="similarity",         # 질문과 벡터가 가까운 순서로 검색
    search_kwargs={"k": TOP_K},
)
print(" 벡터 저장소 및 검색기 설정 완료\n")


# Step 4: 검색된 문서를 프롬프트에 넣을 문자열로 변환
def format_docs(docs):
    """검색된 청크에 번호와 출처를 붙여 LLM 이 구분하기 쉽게 합칩니다."""
    parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "알 수 없음")
        parts.append(f"[문서 {i}]\n출처: {source}\n내용: {doc.page_content.strip()}")
    return ("\n" + "=" * 50 + "\n").join(parts)  # 문서 사이에 구분선 삽입


# Step 5: 프롬프트 템플릿 ({context}, {question} 자리에 값이 채워짐)
print(" Step 4: 프롬프트 설정...")
template = '''당신은 축구 규칙 전문가입니다. 아래 제공된 컨텍스트를 바탕으로 사용자의 질문에 정확하고 상세하게 답변해주세요.

**답변 지침:**
1. 제공된 컨텍스트의 정보만을 사용하여 답변하세요
2. 구체적인 수치, 규칙, 조건들을 포함하여 상세히 설명하세요
3. 관련된 여러 규칙이 있다면 체계적으로 정리해주세요
4. 컨텍스트에 없는 정보는 추측하지 마세요

**컨텍스트:**
{context}

**질문:** {question}

**답변:**
위 컨텍스트를 바탕으로 질문에 대해 상세히 답변드리겠습니다.

'''
prompt = ChatPromptTemplate.from_template(template)

# Step 6: LLM 설정
print(" Step 5: LLM 모델 설정...")
model = ChatUpstage(
    model=CHAT_MODEL,
    base_url=UPSTAGE_BASE_URL,
    temperature=0.5,
)

# Step 7: RAG 체인 구성
print(" Step 6: RAG 체인 구성...")
# 답변 생성 체인: {"context": ..., "question": ...} → 프롬프트 → LLM → 문자열
generate_chain = prompt | model | StrOutputParser()
# 전체 체인: 질문 → (검색 → format_docs → context) + (질문 그대로 → question) → 답변 생성
# rag_chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | generate_chain
rag_chain = (
    {
        # 입력(질문 문자열)이 retriever 로 들어가 관련 청크 리스트를 반환하고,
        # format_docs 가 이를 하나의 문자열로 합쳐 프롬프트의 {context} 에 채움
        "context": retriever | format_docs,
        # 입력 질문을 변형 없이 그대로 전달해 프롬프트의 {question} 에 채움
        "question": RunnablePassthrough(),
    }  # dict 는 파이프(|)로 연결되면 RunnableParallel 로 변환되어 두 값을 같은 입력으로 동시에 계산
    | generate_chain  # {"context": ..., "question": ...} 를 받아 프롬프트 → LLM → 문자열 순으로 처리
)

print(" RAG 파이프라인 설정 완료!\n")

In [ ]:
query = "선수의 장비에 대하여 설명해 주세요."

response = rag_chain.invoke(query)  

# 최종 응답 출력
print(f" 모델 응답:\n")
pprint(response)

In [ ]:
# Step 8: 테스트 실행
def ask_question(question):
    """질문을 받아 RAG 시스템으로 답변 생성 (검색은 1회만 수행)"""
    print(f" 질문: {question}")
    print("-" * 60)

    # 관련 문서를 먼저 검색해 개수 확인
    docs = retriever.invoke(question)
    print(f" 검색된 관련 문서 수: {len(docs)}")

    # 이미 검색한 문서로 답변 생성 (rag_chain.invoke 를 쓰면 검색이 한 번 더 실행됨)
    response = generate_chain.invoke({"context": format_docs(docs), "question": question})

    print(" 답변:")
    print(response)
    print("\n" + "=" * 80 + "\n")

    return response


# 테스트 질문들
if __name__ == "__main__":
    test_questions = [
        "경기장 표시에 대해서 설명해주세요.",
        "페널티 에어리어의 크기와 규격은 어떻게 되나요?",
        "오프사이드 규칙에 대해 자세히 설명해주세요.",
        "축구공의 규격과 요구사항은 무엇인가요?",
        "골키퍼가 할 수 없는 행동들은 무엇인가요?",
    ]

    for question in test_questions:
        ask_question(question)